# Nível 1 — Parte A: dados e regras determinísticas

Nesta etapa, os cálculos são feitos exclusivamente com pandas. A LLM será usada apenas na Parte B.

In [1]:
import json
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
DATA_PATH = Path('../dados/dados_nivel_1.json')
dados = json.loads(DATA_PATH.read_text(encoding='utf-8'))
TAXA_USD_BRL = dados['taxa_cambio_usd_brl']
df_bruto = pd.DataFrame(dados['operacoes'])
print(f'Taxa fixa usada: R$ {TAXA_USD_BRL:.2f} por USD')
print(f'Registros carregados: {len(df_bruto)}')

Taxa fixa usada: R$ 5.40 por USD
Registros carregados: 20


## 1. Diagnóstico da qualidade dos dados

Verificamos duplicidades, datas ausentes, moedas, valores inválidos e campos ausentes antes de aplicar as regras.

In [2]:
campos_esperados = {'id', 'cliente_id', 'data', 'valor', 'moeda', 'canal', 'tipo', 'contraparte', 'observacao'}
ids_duplicados = df_bruto[df_bruto['id'].duplicated(keep=False)].sort_values('id')
datas_ausentes = df_bruto[df_bruto['data'].isna()]
campos_ausentes = sorted(campos_esperados - set(df_bruto.columns))
valores_invalidos = df_bruto[df_bruto['valor'].isna() | (df_bruto['valor'] <= 0)]
print('IDs duplicados:')
print(ids_duplicados[['id', 'cliente_id', 'valor', 'contraparte']].to_string(index=False))
print(f'\nDatas ausentes: {datas_ausentes["id"].tolist()}')
print(f'Moedas encontradas: {df_bruto["moeda"].value_counts().to_dict()}')
print(f'Campos ausentes: {campos_ausentes}')
print(f'Valores nulos ou não positivos: {len(valores_invalidos)}')

IDs duplicados:
     id cliente_id  valor         contraparte
OP-0007    CLI-A-3  17200 Epsilon Consultoria
OP-0007    CLI-A-3  17200 Epsilon Consultoria

Datas ausentes: ['OP-0017']
Moedas encontradas: {'BRL': 19, 'USD': 1}
Campos ausentes: []
Valores nulos ou não positivos: 0


### Decisões de limpeza

- `OP-0007` aparece duas vezes com os mesmos campos; removo a segunda ocorrência usando `id` como chave de unicidade.
- `OP-0017` não tem data, mas contém valor e observação explicando a falha do sistema; preservo o registro. Ele não participa da Regra 1, que depende de data.
- A operação em USD é preservada e convertida para BRL pela taxa fixa fornecida.
- Não há campos ausentes nem valores nulos ou não positivos que exijam descarte adicional.

In [3]:
df = df_bruto.copy()
for coluna in ['id', 'cliente_id', 'moeda', 'canal', 'tipo', 'contraparte', 'observacao']:
    df[coluna] = df[coluna].fillna('').astype(str).str.strip()
df = df.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
df['valor_brl'] = df['valor'] * df['moeda'].map({'BRL': 1.0, 'USD': TAXA_USD_BRL})
df['data_ausente'] = df['data'].isna()
print(f'Registros após deduplicação: {len(df)}')
print(f'Registros preservados sem data: {int(df["data_ausente"].sum())}')
df.head()

Registros após deduplicação: 19
Registros preservados sem data: 1


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,data_ausente
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,"18,100.00",False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,"17,300.00",False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,"18,800.00",False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,"3,300.00",False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,"25,900.00",False


## 2. Agregações descritivas

As duas agregações pedidas são calculadas no DataFrame limpo e usam `valor_brl`.

In [4]:
volume_por_cliente = (df.groupby('cliente_id', as_index=False)['valor_brl'].sum().rename(columns={'valor_brl': 'volume_total_brl'}).sort_values('volume_total_brl', ascending=False))
quantidade_por_canal = (df.groupby('canal', as_index=False)['id'].count().rename(columns={'id': 'quantidade_operacoes'}).sort_values('quantidade_operacoes', ascending=False))
print('Volume total por cliente:')
display(volume_por_cliente)
print('Quantidade de operações por canal:')
display(quantidade_por_canal)

Volume total por cliente:


,cliente_id,volume_total_brl
3,CLI-A-4,"79,500.00"
0,CLI-A-1,"57,500.00"
1,CLI-A-2,"52,900.00"
2,CLI-A-3,"48,500.00"
4,CLI-A-5,"16,900.00"
5,CLI-A-6,"10,200.00"


Quantidade de operações por canal:


,canal,quantidade_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


## 3. Regra 1 — fracionamento

Sinalizamos todas as operações de um cliente em uma mesma data quando o grupo tem pelo menos três operações, soma superior a R$ 50.000,00 e nenhuma operação individual atinge R$ 20.000,00. Grupos sem data são excluídos porque não é possível estabelecer a mesma data com segurança.

In [5]:
grupos_por_data = (df.dropna(subset=['data']).groupby(['cliente_id', 'data'], as_index=True).agg(quantidade_operacoes=('id', 'size'), soma_brl=('valor_brl', 'sum'), max_operacao_brl=('valor_brl', 'max')))
grupos_fracionamento = grupos_por_data[(grupos_por_data['quantidade_operacoes'] >= 3) & (grupos_por_data['soma_brl'] > 50000) & (grupos_por_data['max_operacao_brl'] < 20000)]
chaves = pd.MultiIndex.from_frame(df[['cliente_id', 'data']])
df['regra_1_fracionamento'] = chaves.isin(grupos_fracionamento.index)
print('Grupos que atendem à Regra 1:')
display(grupos_fracionamento.reset_index())
print('Operações sinalizadas:')
display(df.loc[df['regra_1_fracionamento'], ['id', 'cliente_id', 'data', 'valor_brl']])

Grupos que atendem à Regra 1:


,cliente_id,data,quantidade_operacoes,soma_brl,max_operacao_brl
0,CLI-A-1,2026-03-09,3,"54,200.00","18,800.00"


Operações sinalizadas:


,id,cliente_id,data,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,"18,100.00"
1,OP-0002,CLI-A-1,2026-03-09,"17,300.00"
2,OP-0003,CLI-A-1,2026-03-09,"18,800.00"


## 4. Regra 2 — valor atípico

Calculamos a mediana por cliente e sinalizamos operações acima de cinco vezes essa mediana. A regra só é aplicada a clientes com pelo menos quatro operações. Todos esses cálculos são feitos em pandas.

In [6]:
df['quantidade_cliente'] = df.groupby('cliente_id')['id'].transform('size')
df['mediana_cliente_brl'] = df.groupby('cliente_id')['valor_brl'].transform('median')
df['limite_atipico_brl'] = 5 * df['mediana_cliente_brl']
df['regra_2_valor_atipico'] = ((df['quantidade_cliente'] >= 4) & (df['valor_brl'] > df['limite_atipico_brl']))
display(df.loc[df['regra_2_valor_atipico'], ['id', 'cliente_id', 'valor_brl', 'mediana_cliente_brl', 'limite_atipico_brl']])

,id,cliente_id,valor_brl,mediana_cliente_brl,limite_atipico_brl
12,OP-0013,CLI-A-4,"64,800.00","5,450.00","27,250.00"


## 5. Validação das regras

A Regra 1 deve capturar `CLI-A-1` em 2026-03-09: três operações somam R$ 54.200,00 e todas são menores que R$ 20.000,00. `CLI-A-3` na mesma data é parecido, mas soma R$ 48.500,00 e não deve ser sinalizado.

In [7]:
validacao_regra_1 = pd.DataFrame([
    {'caso': 'esperado positivo', 'cliente_id': 'CLI-A-1', 'data': '2026-03-09', 'soma_esperada_brl': 54200.00, 'sinalizado': bool(df.loc[(df['cliente_id'] == 'CLI-A-1') & (df['data'] == '2026-03-09'), 'regra_1_fracionamento'].any())},
    {'caso': 'parecido, mas abaixo do limite', 'cliente_id': 'CLI-A-3', 'data': '2026-03-05', 'soma_esperada_brl': 48500.00, 'sinalizado': bool(df.loc[(df['cliente_id'] == 'CLI-A-3') & (df['data'] == '2026-03-05'), 'regra_1_fracionamento'].any())},
 ])
display(validacao_regra_1)
assert bool(validacao_regra_1.loc[0, 'sinalizado']) is True
assert bool(validacao_regra_1.loc[1, 'sinalizado']) is False
print('Validação da Regra 1 concluída com sucesso.')

,caso,cliente_id,data,soma_esperada_brl,sinalizado
0,esperado positivo,CLI-A-1,2026-03-09,"54,200.00",True
1,"parecido, mas abaixo do limite",CLI-A-3,2026-03-05,"48,500.00",False


Validação da Regra 1 concluída com sucesso.


In [8]:
resumo_sinalizacoes = df[['id', 'cliente_id', 'data', 'valor_brl', 'regra_1_fracionamento', 'regra_2_valor_atipico']].copy()
resumo_sinalizacoes['quantidade_regras'] = resumo_sinalizacoes[['regra_1_fracionamento', 'regra_2_valor_atipico']].sum(axis=1)
display(resumo_sinalizacoes[resumo_sinalizacoes['quantidade_regras'] > 0])

,id,cliente_id,data,valor_brl,regra_1_fracionamento,regra_2_valor_atipico,quantidade_regras
0,OP-0001,CLI-A-1,2026-03-09,"18,100.00",True,False,1
1,OP-0002,CLI-A-1,2026-03-09,"17,300.00",True,False,1
2,OP-0003,CLI-A-1,2026-03-09,"18,800.00",True,False,1
12,OP-0013,CLI-A-4,2026-03-24,"64,800.00",False,True,1
